# 🎬 yt-dlp Studio v2 — Google Colab

Professional video downloader with skeuomorphic Web UI.

### Quick Start
1. **(Optional)** Set your GitHub repo URL in Cell 1
2. Run all cells in order (Cell 1 → Cell 5)
3. Click the printed URL after Cell 5 runs

### Features
- 🎥 Video + codec/resolution picker (AV1, VP9, H.264, H.265…)
- 🎵 Audio-only extraction (MP3, M4A, Opus, FLAC, WAV)
- 🔇 Silent video (no audio track)
- 📦 MKV with multiple audio + subtitle tracks
- ✂️ Time-range clip download
- 💬 Subtitle selection & embedding
- 📁 One-click file download to your device
- 🚫 SponsorBlock, chapter splitting, cookie support
- 📁 Optional Google Drive output


In [ ]:
# ── CELL 1 ── Config & optional Google Drive
import os

# ─────────────────────────────────────────────────────────────────
# If you push this project to GitHub, set your repo URL here.
# The notebook will pull files from it instead of writing them inline.
# Format: 'https://github.com/YOUR_USER/YOUR_REPO'
# Leave as '' to use the bundled inline files.
GITHUB_REPO = ''  # e.g. 'https://github.com/alice/ytdlp-studio'
GITHUB_BRANCH = 'main'
# ─────────────────────────────────────────────────────────────────

USE_GDRIVE = True  # Set False to save to /content/downloads instead

if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DOWNLOAD_DIR = '/content/drive/MyDrive/yt-dlp-downloads'
else:
    DOWNLOAD_DIR = '/content/downloads'

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

with open('/tmp/cfg.txt', 'w') as f:
    f.write(f'{DOWNLOAD_DIR}\n{GITHUB_REPO}\n{GITHUB_BRANCH}')

print(f'Download dir : {DOWNLOAD_DIR}')
print(f'GitHub repo  : {GITHUB_REPO or "(inline mode)"}')


In [ ]:
# ── CELL 2 ── Install dependencies
print('Installing system dependencies...')
!apt-get install -qq -y ffmpeg nodejs npm > /dev/null 2>&1
print('Installing yt-dlp...')
!pip install -q yt-dlp
print('All system deps ready.')


In [ ]:
# ── CELL 3 ── Pull project files from GitHub or write inline
import os, subprocess

with open('/tmp/cfg.txt') as f:
    lines = f.read().splitlines()
    DOWNLOAD_DIR = lines[0]
    GITHUB_REPO  = lines[1] if len(lines) > 1 else ''
    GITHUB_BRANCH= lines[2] if len(lines) > 2 else 'main'

APP_DIR = '/content/ytdlp-studio'
os.makedirs(APP_DIR + '/public', exist_ok=True)

if GITHUB_REPO:
    print(f'Pulling from GitHub: {GITHUB_REPO}')
    raw_base = GITHUB_REPO.replace('github.com', 'raw.githubusercontent.com') + f'/{GITHUB_BRANCH}'
    import urllib.request
    files_to_fetch = [
        ('server/package.json',  f'{APP_DIR}/package.json'),
        ('server/server.js',     f'{APP_DIR}/server.js'),
        ('public/index.html',    f'{APP_DIR}/public/index.html'),
    ]
    for rel, dest in files_to_fetch:
        url = f'{raw_base}/{rel}'
        print(f'  Fetching {rel}...')
        urllib.request.urlretrieve(url, dest)
    print('All files fetched from GitHub.')
else:
    print('No GitHub repo set — using inline bundled files.')
    # Write package.json inline
    pkg = '{"name":"ytdlp-studio","version":"2.0.0","dependencies":{"express":"^4.18.2","express-ws":"^5.0.2","multer":"^1.4.5-lts.1","uuid":"^9.0.0","cors":"^2.8.5"}}'
    with open(f'{APP_DIR}/package.json', 'w') as f: f.write(pkg)
    # server.js and index.html were written by previous cells (see GitHub workflow)
    print('NOTE: For inline mode, you also need to run the write-server and write-UI cells.')
    print('The recommended workflow is to push to GitHub and set GITHUB_REPO above.')

print('Installing Node packages...')
result = subprocess.run(['npm', 'install', '--silent'], cwd=APP_DIR, capture_output=True, text=True)
if result.returncode != 0:
    print('npm error:', result.stderr[:500])
else:
    print('npm install OK.')


In [ ]:
# ── CELL 4 ── (Optional) Write server.js + index.html inline
# Only needed if you are NOT using a GitHub repo.
# If GITHUB_REPO is set in Cell 1, skip this cell.
import os

with open('/tmp/cfg.txt') as f:
    GITHUB_REPO = f.read().splitlines()[1] if len(f.read().splitlines()) > 1 else ''

# Re-read since we already consumed f
with open('/tmp/cfg.txt') as f:
    cfg = f.read().splitlines()
GITHUB_REPO = cfg[1] if len(cfg) > 1 else ''

if GITHUB_REPO:
    print('GitHub repo is set — skipping inline write. Files already fetched in Cell 3.')
else:
    print('Writing inline files...')
    print('Please copy server/server.js and public/index.html from the ZIP')
    print('into /content/ytdlp-studio/ and /content/ytdlp-studio/public/ respectively,')
    print('OR set GITHUB_REPO in Cell 1 to pull automatically.')
    print()
    print('Tip: Upload the ZIP to Colab, unzip it, then copy files:')
    print('  !unzip ytdlp-studio.zip -d /tmp/ytdlp-src')
    print('  !cp /tmp/ytdlp-src/server/server.js /content/ytdlp-studio/')
    print('  !cp /tmp/ytdlp-src/public/index.html /content/ytdlp-studio/public/')


In [ ]:
# ── CELL 5 ── Launch server & open tunnel
import subprocess, threading, time, os

with open('/tmp/cfg.txt') as f:
    DOWNLOAD_DIR = f.read().splitlines()[0]

APP_DIR = '/content/ytdlp-studio'

# Kill any existing server
!pkill -f 'node.*server.js' 2>/dev/null || true
time.sleep(1)

env = os.environ.copy()
env['DOWNLOAD_DIR'] = DOWNLOAD_DIR
env['PORT'] = '3000'

srv = subprocess.Popen(
    ['node', 'server.js'],
    cwd=APP_DIR, env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

def _log():
    for line in srv.stdout:
        print('[server]', line.decode().rstrip())
threading.Thread(target=_log, daemon=True).start()
time.sleep(2)

from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(3000)')

print()
print('━' * 60)
print('  yt-dlp Studio v2 is LIVE')
print()
print(f'  🔗 {url}')
print(f'  📂 {DOWNLOAD_DIR}')
print('━' * 60)
print('Keep this cell running. Re-run to restart the server.')

srv.wait()
